In [3]:
import requests
import time

BASE_URL = "http://localhost:9000"


def test_controller():
    print("\n🚀 STEP 1: SUBMIT TASK")

    res = requests.post(
        f"{BASE_URL}/execute",
        params={"query": "calculate interest for 1000"}
    )
    data = res.json()
    print("Submit Response:", data)

    if "task_id" not in data:
        raise Exception(f"❌ Failed to submit task: {data}")

    task_id = data["task_id"]
    print("🆔 Task ID:", task_id)


    print("\n⏳ STEP 2: POLLING")

    while True:
        time.sleep(2)

        res = requests.get(f"{BASE_URL}/result/{task_id}")
        data = res.json()
        print("Status:", data)

        # =========================
        # 🔥 APPROVAL HANDLING (FIXED)
        # =========================
        if data["status"] == "pending_approval":
            print("\n🧠 APPROVAL REQUIRED")
            print(data["message"])

            # 🔥 User input
            user_input = input("👉 Approve? (yes/no): ").strip().lower()

            # 🔥 Mapping FIX (CRITICAL)
            mapping = {
                "yes": "approve",
                "y": "approve",
                "approve": "approve",
                "no": "reject",
                "n": "reject",
                "reject": "reject"
            }

            decision = mapping.get(user_input, "reject")

            print(f"➡ Sending decision: {decision}")

            approve_res = requests.post(
                f"{BASE_URL}/approve/{task_id}",
                json={"decision": decision}   # ✅ IMPORTANT
            )

            print("👍 Approval Response:", approve_res.json())

        # =========================
        # ✅ COMPLETED
        # =========================
        elif data["status"] == "completed":
            print("\n✅ FINAL RESULT:", data)
            break

        # =========================
        # ❌ FAILED / REJECTED
        # =========================
        elif data["status"] in ["failed", "rejected"]:
            print("\n❌ FINAL STATUS:", data)
            break




In [5]:

# 🔥 Run test
test_controller()


🚀 STEP 1: SUBMIT TASK
Submit Response: {'task_id': 'c7ed10f3-2416-4cef-b721-46ad22cada33', 'status': 'waiting_for_approval', 'approval_message': 'Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?'}
🆔 Task ID: c7ed10f3-2416-4cef-b721-46ad22cada33

⏳ STEP 2: POLLING
Status: {'status': 'pending_approval', 'message': 'Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?'}

🧠 APPROVAL REQUIRED
Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?


👉 Approve? (yes/no):  no


➡ Sending decision: reject
👍 Approval Response: {'status': 'rejected', 'message': 'Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?'}
Status: {'status': 'rejected', 'message': 'Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?'}

❌ FINAL STATUS: {'status': 'rejected', 'message': 'Do you want to CALCULATE INTEREST for ₹1000 at 5% for 2 years?'}
